In [16]:
import pandas as pd
pd.set_option("display.max_rows", 1000)

In [70]:
from preprocess import load_data

data = load_data()

data.keys()

dict_keys(['course_chapter_items', 'courses', 'subgroups', 'test_seen', 'test_seen_group', 'test_unseen', 'test_unseen_group', 'train', 'train_group', 'users', 'val_seen', 'val_seen_group', 'val_unseen', 'val_unseen_group'])

In [4]:
data['courses'].iloc[0:1]

,course_id,course_name,course_price,teacher_id,teacher_intro,groups,sub_groups,topics,course_published_at_local,description,will_learn,required_tools,recommended_background,target_group
0,61888e868f154b000781b45a,少女人妻華麗變身：七大妝容七彩的夢幻樂園,1800,61888e7bb7fe1c0006850eff,在美妝 KOL 圈裡屬個人風格強烈的 Alice，在清新與叛逆風格間遊刃有餘，其幽默的美妝影...,生活品味,"更多生活品味,護膚保養與化妝","更多生活品味,護膚保養與化妝",NaN,"<img src=""https://images.api.hahow.in/images/6...",不再害怕各種顏色的彩妝，可以更隨心搭配各種繽紛的顏色。\n,所需工具為：視課程實際會用到的彩妝用品,只要你有一顆愛化妝、想變漂亮的心皆可以參加。\n\n⚠️ 雖然課程當中會帶到相關彩妝技巧，不...,熱愛彩妝的人


In [5]:
data['users'].iloc[0:1]

,user_id,gender,occupation_titles,interests,recreation_names
0,54ccaa73a784960a00948687,female,NaN,"職場技能_創業,藝術_電腦繪圖,設計_介面設計,設計_動態設計,設計_平面設計,投資理財_投...",NaN


### process courses

In [6]:
data['train'].iloc[1]['course_id']

'5bfd47782d018e0020e4b0e4 5fc4a352d375951a03cc0d45 6090dda489d06d6a564c9a78 5fd9b1ce0fb8aa8b32928d5b 6030c9cd99e14cc2401e66b9 5cad79b0d133060020f44c5e 5f1f9092673f8866bf9604af 5f59ff38f24eac1485f8097f 607015a2db27b36747727e80'

In [7]:
data['train'].query("user_id == '56dae2b74e3ef90900b7bd0e'")

,user_id,course_id
31292,56dae2b74e3ef90900b7bd0e,5fc5ee1b08b74a6e3723abd2 6059aee039f2512548c18...


In [8]:
data['val_unseen']

,user_id,course_id
0,612c1fcd560d8100069aa5ba,5f14aeabcad0d0afe5ea3898 60ac8e42992b7aadf9d019d8
1,558b954c6dec460f00111131,5fc5ed671be929537e5283bb 5fc5edae001c9102feab8...
2,591de47900f58c070078cd99,60aeac37bca91777bf5bb114
3,612e5ebd31e50c0006fc4611,5fc5ee1b08b74a6e3723abd2 60cb0a440dabda80019d5...
4,60c248c62757c16af6160429,60a49d7c62c6aea8d38e3903
...,...,...
11617,617eb7aff2c5f00006592046,6030c9cd99e14cc2401e66b9 5fc5ee1b08b74a6e3723a...
11618,613253cdd5d02a0007225fe4,5e8e9d3c5a140c3d1e1c5245
11619,617eb70db23194000781f7cf,61381a3f33a3960006df1eb3
11620,5c9f3af172a72f0021508a87,5ce3a183138a45002046859d


In [10]:
data['train']['course_id'] = data['train']['course_id'].apply(lambda s: s.split())

In [11]:
train_df = data['train'].explode("course_id")

In [12]:
courses = sum(data['val_unseen'].course_id.apply(lambda s: s.split()).to_list(), [])

In [13]:
len(courses)

19618

In [14]:
df_tmp = data['courses'].merge(df, on=["course_id"], indicator = True, how='left').loc[lambda x : x['_merge']!='both']

NameError: name 'df' is not defined

In [15]:
train_df.groupby("course_id").count()

,user_id
course_id,
54d5a117065a7e0e00725ac0,60
54d5d9952246e60a009ec571,11
54d7148a2246e60a009ec588,55
54f1268f4ec3c809002e4a29,23
551171a938239d1000577864,13
...,...
60f83c44b355920006abbd3f,54
60fa9c7d1917300007734fd3,86
60ffeb101f74d40006fe3fd8,8


In [185]:
data['val_unseen'].course_id.apply(lambda s: s.split()).explode()

0        5f14aeabcad0d0afe5ea3898
0        60ac8e42992b7aadf9d019d8
1        5fc5ed671be929537e5283bb
1        5fc5edae001c9102feab8ecf
1        5e55f57bc8bfb600249406e1
                   ...           
11617    5ef099ab678184065fd4d426
11618    5e8e9d3c5a140c3d1e1c5245
11619    61381a3f33a3960006df1eb3
11620    5ce3a183138a45002046859d
11621    6155cda6d425f500065f5c96
Name: course_id, Length: 19618, dtype: object

In [17]:
pd.to_datetime(data['courses'].course_published_at_local).max()

Timestamp('2022-03-29 12:00:16.763000')

In [18]:
pd.to_datetime(data['courses'].merge(data['val_seen'].course_id.apply(lambda s: s.split()).explode(), on=["course_id"], how="inner")["course_published_at_local"]).max()

Timestamp('2022-01-27 12:02:16.752000')

In [19]:
pd.to_datetime(data['courses'].course_published_at_local).max()

Timestamp('2022-03-29 12:00:16.763000')

In [161]:
data['courses'].groupby(["teacher_id", "course_id"]).agg(n=("course_id", "count")).sort_values("n").reset_index().merge(train_df.groupby("course_id").count(), on=["course_id"], how="left").fillna(0).set_index(["teacher_id", "course_id"]).sort_index()

,,n,user_id
teacher_id,course_id,,
54cb5c2ea784960a00948678,54d5d9952246e60a009ec571,1,11.0
54d5a079065a7e0e00725abe,54d5a117065a7e0e00725ac0,1,60.0
54d713532246e60a009ec586,54d7148a2246e60a009ec588,1,55.0
54de6f88dbc18009004c1605,54f1268f4ec3c809002e4a29,1,23.0
54e4b5fbc5c9c00900cd8d4c,59b8a97dcd5792070055ef75,1,291.0
54f5c694e977de0a00d46a06,551171a938239d1000577864,1,13.0
54f810e200a1420f0068283a,5593f992cfe8320b00ccd4c4,1,27.0
54fe479ab27eaa0b0024edd1,5f506991b1b6d678526889e1,1,25.0
5511977138239d100057786e,551198a738239d1000577870,1,32.0


In [191]:
from datetime import datetime

In [131]:
data['val_unseen'].course_id.apply(lambda s: s.split()).explode().to_frame("course_id").groupby("course_id").agg(n=("course_id", "count")).merge(df_tmp, on=["course_id"]).sort_values("n_x")

,course_id,n_x,course_name,course_price,teacher_id,teacher_intro,groups,sub_groups,topics,course_published_at_local,description,will_learn,required_tools,recommended_background,target_group,n_y,_merge
0,58bfdaab5c4e6507007cdeca,1,APP Inventor 2 進階實戰,1000,58650351f108e00800c2507f,行動創客學院 執行長\n長庚大學EMBA兼任助理教授\n師大附中自造實驗室諮詢委員,程式,手機程式開發,App inventor,2017-06-02 16:15:53.234,<h4>「五分鐘程式設計」系列又來囉！</h4><p>107年開始程式設計正式納入課綱，國、...,課程設計是從基礎出發並重視實作，除了能學到各種安卓手機硬體元件基本知識（例如：加速度、指南針...,1. Windows XP或以上/Mac OS（講師將使用 Mac OS）\n2. Chro...,不用對程式設計有經驗，只要有興趣，有一顆熱情學習的心即適合本課程。若有其它任一程式學習經驗更...,1. 希望從零開始學程式\n2. 玩Arduino硬體，並想透過APP遠端控制\n3. 希望...,NaN,left_only
19,611dbe96be4b2d000699b345,1,小資入門籌碼課：提高短線操作勝率,3600,5f8e3a347698f0649cf5da97,豹投資，專業投資人的邏輯整合大數據分析，成為多方投資策略的專業選股系統網站。\n不管在財務面...,投資理財,金融商品,股票,2022-01-25 12:11:38.459,<p>在台股市場做投資，最重要的就是看籌碼。</p><p>在這堂課裡，我們將用科學的方式，論...,1.了解三大法人與市場主力操作特性與思維 \n2.學會台股市場有許多客觀數據，提供你分析股市...,可以上網的裝置，例如筆記型電腦、平板、智慧型手機,特別適合初次接觸籌碼分析，或剛踏入股市想學習籌碼的新鮮人很友善～\n基本只要你對於股市、籌碼...,1. 沒有學過籌碼分析，對投資股票有興趣的新手\n2. 投資股票的時間還不滿一年\n3. 知...,NaN,left_only
1,5fe6fe2c5a5ba942b377c5ec,3,人物專訪怎麼寫？訪綱、採訪、寫作心法,2280,5fe6fdff5a5ba9064377c518,陳芷儀，耳草人內容工作室創辦人暨內容總監。天秤座 O 型（如果有人在意），政大傳播所畢。寫各...,人文,文學,文學,2022-01-24 12:08:59.923,<p>網站、雜誌、報紙⋯⋯，閱遍各類媒體，人物專訪絕對是不可或缺的元素。一篇專訪，可以幫助我...,1. 能規劃一場人物專訪\n2. 能擬定具水準的訪綱\n3. 在專訪現場的心理準備及臨場反應...,1. 電腦\n2. Google 雲端功能（建議但非必要）,1. 初學：完全沒有採訪經驗，想知道如何開始。\n2. 中階：有些採訪相關經驗，想持續進步。...,1. 對人物專訪或新聞採訪有興趣\n2. 想加強一對一、一對多對談的口語溝通技巧\n3. 想...,NaN,left_only
31,61666a458fc5c300073e6f60,5,放下酒譜！跟著 Mars 飛向浩瀚無垠調酒宇宙,3980,61551818a51e9e0007011eaa,【Mars】 為「2021年World Class世界頂尖調酒大賽」台灣冠軍。Mars自學習...,生活品味,烹飪料理與甜點,更多烹飪,2022-01-26 12:00:37.670,<blockquote>一堂超過 6 小時的全系列調酒課程，<br />5 道經驗淬煉的風味...,★ 5 道冠軍調酒師經驗淬煉的風味公式 ★ 學會超過百種排列組合的雞尾酒\n�� 快速學...,．只需要任何一款基酒跟一個結構，就可以調出數十杯以上的雞尾酒\n．糖漿，課程中也會教您自製簡...,．適合有基礎調酒概念者，可學會更獨具創意的雞尾酒以及概念發想\n．若為雞尾酒新手，建議從第一...,．適合曾多次製作過，但始終不知道怎麼調整細節讓調酒更好的人。\n．適合想嘗試做雞尾酒，但還沒...,NaN,left_only
26,6135374d94b8350007f7fe43,8,植物水彩繪 - 石斛蘭的觀察紀錄,2500,5fd30f93a79fe6299ab582f0,「植物藝術工作室』臉書粉專版主。雖然在城市長大，但更嚮往鄉下的生活。於是畫畫便是我療癒自我的...,藝術,繪畫與插畫,水彩,2022-01-18 12:00:26.742,"<img alt src=""https://images.api.hahow.in/imag...",1. 可以從眾家廠牌中挑選出適合植物水彩繪圖的工具\n2. 瞭解蘭花構造及相關知識\n3. ...,1. 圓頭水彩筆：建議筆毛要有彈性且不能太硬。\n2. 條狀或塊狀透明水彩顏料：學生級或是專...,喜歡植物，喜歡畫圖的學生都很適合。相信你不論是否已有植物或繪圖相關經驗，都可以從這門課吸收到...,1. 喜歡植物也想要把植物的美記錄下來的人\n2. 喜歡水彩也希望可以循序漸進進入水彩領域的...,NaN,left_only
13,60ed88d6b89d2300069ee963,9,新手開店0到1！餐飲創業成功必讀攻略術！,3980,60520446e928c7d51a1958d8,617行銷筆記，是為協助有心在商業行銷領域精進的人，準備的行銷觀點整理。\n \n617創立...,職場技能,創業,自主創業,2021-12-01 12:00:04.051,"<img alt src=""https://images.api.hahow.in/imag...",1.提高餐飲業的基礎知識及未來趨勢\n2.建立餐飲創業正確觀念，打好穩健經營基礎\n3.瞭解...,課程無需使用到任何工具/軟體，只要有一顆學習的心就好！,此課程主要設計給餐飲業「創業新手」、「經營一段時間但遇到瓶頸或找不到方法」的初學者\n不需具...,此課程較適合「初次創業餐飲店」的人：\n1.想開餐飲當老闆的人\n2.想開餐飲店卻還在籌備階...,NaN,left_only
2,5ff6aa28c5cbbcb694532eaa,10,插一盆花歌頌四季－我的自然感花藝創作法,2350,5cc6b925bc5432002078bd83,PM FLOWERS 創辦人、花藝設計師、花藝老師\nhttps://www.instagr...,"生活品味,藝術","壓力舒緩,更多生活品味,更多藝術","壓力舒緩,更多藝術",2021-11-30 12:00:02.723,"<img alt src=""https://images.api.hahow.in/imag...",1. 了解如何用更環保的方式插花，告別“花藝海綿”。\n2. 更細微地感受素材，了解他們在一...,必要工具：\n花藝剪刀、鐵絲雞網、劍山、黏土、膠帶與膠帶台、防水花瓶（透明、非透明皆可）、鐵...,只要對於花草充滿愛與熱誠就可以了！歡迎所有初學者喔,1. 想在家信手捻來享受插花美好的你。\n2. 已經成為花藝師或在花店工作，但想了解除了使...,NaN,left_only
25,6130495cd5d02a00071f2c3b,12,PChome 電商大學：給初學者的行銷入門課,499,6130492bd5d02a00071f2be5,走過21的本土電商PChome開課囉!希望藉由開創台灣電商藍圖的經驗，分享給對電商產業有興趣...,行銷,數位行銷,數位行銷,2022-01-05 12:01:22.835,"<p></p><img alt src=""https://images.api.hahow....",1.了解電商行銷的線上線下布局，如檔期策略、異業合作到話題擴散\n2.認識電商行銷在社群經營...,一顆對於電商產業有興趣及熱情、熱忱的心,初學者，歡迎完全不懂電商，但想要了解電商經營與行銷，但卻不知從何入門的大學生、社會新鮮人、轉...,1.\t大學在學生\n2.\t社會新鮮人\n3.\t對電商有興趣之創業者\n\n不適合：已經...,NaN,left_only
16,60fe7a7e006b74000621ce1f,13,雙語教學超前部署！互動式英語授課技巧,2680,5afa5a0ebb4174001e3fe107,CLN 致力於引領企業用外語佈局全球、教導學員以外語提升競爭力，讓客戶自在使用外語的同時，亦...,職場技能,更多職場技能,教學技巧,2022-01-04 12:00:02.459,<h4>&lt;試看單元搶先看&gt;</h4><blockquote>1-1 為什麼會開這...,(1) 陳述雙語教學與全英語教學的差異\n(2) 使用 CLIL (Content and ...,筆記本和一支筆，或是任何可以幫助你思考、實作紀錄的用品皆可,(1) 具備 CEFR B1 等級之英語能力分級測驗\n(2) 全民英檢中級或多益 650 ...,(1) 不分科目、年級，對雙語教學有熱忱的體制內老師、師培生\n(2) 不分程度、領域，想嘗...,NaN,left_only
28,613c4d77323c7f00

In [205]:
t = data['val_seen'].set_index("user_id").course_id.apply(lambda s: s.split()).explode().to_frame("course_id")

In [20]:
user_course_dict = data['train'].set_index("user_id")["course_id"].to_dict()

In [26]:
df = train_df["course_id"].value_counts().to_frame("n")

In [34]:
df = df.reset_index().rename(columns={"index": "course_id"})

In [35]:
df_ = data['val_seen'].course_id.apply(lambda s: s.split()).explode().to_frame("course_id")
for i in range(100, 2001, 50):
    popular_course = df.query(f"n > {i}")
    l = len(df_.merge(popular_course, on=["course_id"], how="inner"))
    print(i, len(popular_course), l, l/len(df_))

100 235 9428 0.5291575461637762
150 161 8393 0.47106695852275915
200 123 7743 0.4345849469607678
250 104 7180 0.4029859123309199
300 80 6388 0.3585339843969243
350 72 6202 0.34809451647303136
400 61 5796 0.3253072907896952
450 53 5421 0.3042599764270079
500 50 5305 0.2977493405174833
550 48 5195 0.2915754616377617
600 45 5086 0.2854577089296739
650 40 4974 0.2791715777066846
700 37 4844 0.27187517539428635
750 36 4785 0.26856373126789024
800 36 4785 0.26856373126789024
850 36 4785 0.26856373126789024
900 33 4697 0.2636246281641129
950 31 4557 0.2557669641353763
1000 30 4500 0.25256777235224787
1050 27 4329 0.24297019700286243
1100 23 4204 0.23595442554863333
1150 23 4204 0.23595442554863333
1200 22 4187 0.23500028063085818
1250 21 4051 0.2273671212886569
1300 20 4003 0.22467306505023293
1350 20 4003 0.22467306505023293
1400 20 4003 0.22467306505023293
1450 17 3680 0.20654431161250492
1500 14 3532 0.19823763821069765
1550 12 3344 0.18768591794353706
1600 12 3344 0.18768591794353706
1650

In [42]:
pd.Timestamp('2021-12-21')

Timestamp('2021-12-21 00:00:00')

In [60]:
df = data['train'].explode("course_id")['course_id'].value_counts().to_frame("n").reset_index().rename(columns={"index": "course_id"})

In [65]:
data['courses'].query("n > 0")['course_published_at_local'].max()

Timestamp('2021-11-29 12:00:14.527000')

In [69]:
len(data['courses'])

664

In [67]:
data['courses'].query("course_published_at_local >= @pd.Timestamp('2021-11-29')")

,course_id,course_name,course_price,teacher_id,teacher_intro,groups,sub_groups,topics,course_published_at_local,description,will_learn,required_tools,recommended_background,target_group,n
662,60c2e123dcc8c38df47514c1,雙語教育必學的核心素養－用英文學知識,2000,60c2e0dcc121046b1958594d,新視紀，網羅各專業領域的鬼才，包含業界各個領域，甚至教育界的菁英，一同不藏私的與大家分享一身...,"生活品味,語言","更多生活品味,英文,親子教育","英文,親子教育",2021-11-29 11:51:11.069,"<img alt src=""https://images.api.hahow.in/imag...",● 英文、學科知識、核心素養能力一併提升\n● 提升靈活應用英語的能力\n● 提升獨立思考、...,觀看線上課程的電腦設備、方便做筆記的紙筆。,此本課程不需先備知識，只要有興趣參加即可。\n,適用對象一：國小學生（英文程度小學二年級以上）\n此門課程針對『108 課綱的核心素養』所設...,2
663,5f55fb39b34335d28416bd0c,音樂人必學，編曲製作心法,2199,5ab61884dc2ab6001e266e34,上方房子↑是 ig 可以訊息！\n\nMusic Creator / Youtuber\n—...,音樂,音樂創作,作曲和編曲,2021-11-29 12:00:14.527,"<h4>課程緣起</h4><img alt src=""https://images.api....",1. 將強化本身關於音樂的編曲概念。 \n2.你對任何曲風的音樂，都會有一定的編曲掌握能力。,1.筆記本／數位筆記\n2.電腦／平板／手機（能音樂製作的硬體工具）\n2.DAW 音樂製作...,本堂為中階課程，需有基礎的音樂製作軟體的經驗，但不限軟體廠牌種類，而課程的初衷是希望降低編曲...,因為科技進步的力量，使我們音樂創造方式大大改變。\n1. 剛接觸了數位音樂後但對編曲懵懂的同...,2


In [38]:
data['val_seen'].course_id.apply(lambda s: s.split()).explode()

0       5b61928a8011d1001e356102
1       559e49185850311000fca504
2       60ddc3ca06259d00064c7f17
2       60aeac37bca91777bf5bb114
2       60c84de9eb75ca46e0c25e85
                  ...           
7743    601d03e07e6747ef11bba066
7744    5ca17cd572a72f002150ed9d
7745    61381a3f33a3960006df1eb3
7746    6156a77fdf426a0007cc5fe1
7747    61381a3f33a3960006df1eb3
Name: course_id, Length: 17817, dtype: object

In [83]:
data['val_seen'].course_id.apply(lambda s: s.split()).explode().to_frame("course_id").merge(df, on=["course_id"], how="inner")

,course_id,n
0,5b61928a8011d1001e356102,74
1,5b61928a8011d1001e356102,74
2,5b61928a8011d1001e356102,74
3,5b61928a8011d1001e356102,74
4,5b61928a8011d1001e356102,74
...,...,...
12152,5e9e7c78735b0146e5cbae8b,11
12153,586a2519f108e00800c26e61,3
12154,5e4612145b02f600245ab028,21
12155,5f5052c05a021d2bda2761da,32


In [58]:
len(courses)

17817

In [15]:
data['courses'].query("course_id == '5f59ff38f24eac1485f8097f'")

,course_id,course_name,course_price,teacher_id,teacher_intro,groups,sub_groups,topics,course_published_at_local,description,will_learn,required_tools,recommended_background,target_group
518,5f59ff38f24eac1485f8097f,掌握關鍵數據，黃金、原油投資高勝率全攻略,3600,5a93c4d418b67a001ee39779,「MacroMicro 財經M平方」致力於整合全球總經數據、協助投資人運用經濟指標找到投資方...,投資理財,"投資觀念,金融商品","ETF,投資觀念",2020-12-02 12:00:06.708,<blockquote><b>歡迎登入 Hahow 會員，點選播放視窗右邊的「試看單元」體驗...,原油\n・從 OPEC+及中東地緣政治，看懂哪些國際角力是油價關鍵、哪些又是雜訊 \n・理解...,1. 可以上網的電腦或平板 \n2. 紙筆或是任何可以幫助您紀錄的工具\n3. 推薦輔助工具...,1. 投資商品、投資操作相關名詞的認識\n如：槓桿型 ETF、ETF 折溢價、高收益債券殖利...,我們認為每位投資人都應該了解全球經濟運行， 如果你希望將投資商品觸及全球市場，或者單純想看...


In [38]:
data['val_unseen'].merge(data['train'], on=['user_id'], how='left').dropna()

,user_id,course_id_x,course_id_y
